# Corpus Inventory and OCR

This Google Colab notebook inventories the four election-document groups, preserves the original PDFs, extracts native text, and creates page-addressable OCR derivatives only where extraction is insufficient.

## Evidence handling rules

- Do not modify, rename, or overwrite source PDFs.
- Every output record retains the source-relative path, SHA-256 hash, and page number.
- OCR text is a derivative and must be reviewed against the source image before it supports a substantive claim.
- Redactions, unreadable text, and low-confidence OCR are recorded as limitations, not inferred.
- Do not upload original records to a cloud service unless you are authorized to do so and have reviewed the service's retention and access terms.

In [1]:
# Local Windows setup: activate the workspace .venv, then run this once in a terminal:
# python -m pip install PyMuPDF pytesseract Pillow pandas tqdm
#
# Google Colab setup: uncomment and run the next four lines instead.
# from google.colab import drive
# drive.mount('/content/drive')
# !apt-get -qq update
# !apt-get -qq install -y tesseract-ocr poppler-utils ocrmypdf
# %pip -q install "PyMuPDF==1.27.1" "pytesseract==0.3.13" "Pillow==12.1.1" "pandas==2.3.3" "tqdm==4.67.3"

from pathlib import Path
import hashlib
import json
import re
import shutil
import subprocess
from datetime import datetime, timezone

import fitz
import pandas as pd
import pytesseract
from PIL import Image
from tqdm.auto import tqdm

if shutil.which('tesseract') is None:
    raise RuntimeError('Tesseract is not on PATH. Install Tesseract OCR, then restart the notebook kernel.')
OCR_PDF_COMMAND = shutil.which('ocrmypdf')
print(f'Tesseract: {pytesseract.get_tesseract_version()}')
print('Searchable OCR PDFs:', 'enabled' if OCR_PDF_COMMAND else 'skipped; install OCRmyPDF to enable')

c:\Users\pchri\Documents\ChrisStuff\WhitehouseDotGov\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tesseract: 5.4.0.20240606
Searchable OCR PDFs: skipped; install OCRmyPDF to enable


In [2]:
# In Colab, replace with the folder containing the four numbered source directories.
# SOURCE_ROOT = Path('/content/drive/MyDrive/WhitehouseDotGov')
# SOURCE_ROOT = Path('/content/WhitehouseDotGov')  # Example for manually uploaded sources.
SOURCE_ROOT = Path(r"C:\Users\pchri\Documents\ChrisStuff\WhitehouseDotGov") # To run in laptop

GROUPS = {
    '1. Vulnerabilities in Voting and Counting Systems': 'voting_and_counting_systems',
    '2. ChinaAcquisitionExploitationAmericanVoterData': 'china_voter_data',
    '3. MichiganVoterRegistrationInvestigation': 'michigan_registration_investigation',
    '4. Noncitizens on State Voter Rolls': 'noncitizens_on_state_rolls',
}
OUTPUT_ROOT = SOURCE_ROOT / 'analysis_output'
TEXT_ROOT = OUTPUT_ROOT / 'page_text'
OCR_PDF_ROOT = OUTPUT_ROOT / 'ocr_pdfs'
MIN_NATIVE_TEXT_CHARS = 75
OCR_DPI = 300
OCR_LANGUAGE = 'eng'
LOW_CONFIDENCE = 75.0

missing_groups = [name for name in GROUPS if not (SOURCE_ROOT / name).is_dir()]
if missing_groups:
    raise FileNotFoundError(f'SOURCE_ROOT is incorrect or incomplete. Missing: {missing_groups}')
for directory in (OUTPUT_ROOT, TEXT_ROOT, OCR_PDF_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as source_file:
        for block in iter(lambda: source_file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def document_id(relative_path, source_hash):
    stem = re.sub(r'[^a-z0-9]+', '_', relative_path.stem.lower()).strip('_')
    return f'{stem[:72]}_{source_hash[:12]}'

def compact_text(text):
    return re.sub(r'\s+', ' ', text).strip()

def ocr_page(page):
    pixmap = page.get_pixmap(dpi=OCR_DPI, colorspace=fitz.csRGB, alpha=False)
    image = Image.frombytes('RGB', (pixmap.width, pixmap.height), pixmap.samples)
    data = pytesseract.image_to_data(image, lang=OCR_LANGUAGE, config='--oem 3 --psm 6', output_type=pytesseract.Output.DICT)
    tokens, confidences = [], []
    for token, confidence in zip(data['text'], data['conf']):
        if token.strip():
            tokens.append(token.strip())
            if confidence != '-1':
                confidences.append(float(confidence))
    return ' '.join(tokens), (sum(confidences) / len(confidences) if confidences else 0.0)

In [3]:
pdf_records = []
for folder_name, category in GROUPS.items():
    for pdf_path in sorted((SOURCE_ROOT / folder_name).rglob('*.pdf')):
        if pdf_path.is_file():
            pdf_records.append((pdf_path, category))
if not pdf_records:
    raise RuntimeError('No PDFs found in the configured source folders.')

document_rows, page_rows = [], []
for pdf_path, category in tqdm(pdf_records, desc='Processing PDFs'):
    relative_path = pdf_path.relative_to(SOURCE_ROOT)
    source_hash = sha256_file(pdf_path)
    source_id = document_id(relative_path, source_hash)
    document = fitz.open(pdf_path)
    requires_ocr_pdf = False

    for page_index, page in enumerate(document):
        page_number = page_index + 1
        native_text = compact_text(page.get_text('text'))
        native_chars = len(re.sub(r'\s+', '', native_text))
        method, text, confidence = 'native', native_text, None
        if native_chars < MIN_NATIVE_TEXT_CHARS:
            text, confidence = ocr_page(page)
            text = compact_text(text)
            method = 'ocr'
            requires_ocr_pdf = True

        review_reasons = []
        if method == 'ocr' and confidence is not None and confidence < LOW_CONFIDENCE:
            review_reasons.append('low_ocr_confidence')
        if len(re.sub(r'\s+', '', text)) < MIN_NATIVE_TEXT_CHARS:
            review_reasons.append('low_text_yield')
        page_header = {
            'document_id': source_id,
            'source_relative_path': str(relative_path),
            'source_sha256': source_hash,
            'page_number': page_number,
            'extraction_method': method,
            'ocr_average_confidence': confidence,
        }
        text_directory = TEXT_ROOT / source_id
        text_directory.mkdir(parents=True, exist_ok=True)
        text_path = text_directory / f'page_{page_number:04d}.txt'
        text_path.write_text(json.dumps(page_header, indent=2) + '\n\n' + text + '\n', encoding='utf-8')
        page_rows.append({
            **page_header,
            'native_text_characters': native_chars,
            'extracted_text_characters': len(re.sub(r'\s+', '', text)),
            'review_required': bool(review_reasons),
            'review_reasons': ';'.join(review_reasons),
            'text_file': str(text_path.relative_to(OUTPUT_ROOT)),
        })

    ocr_pdf_path = OCR_PDF_ROOT / f'{source_id}.pdf'
    if requires_ocr_pdf and OCR_PDF_COMMAND:
        result = subprocess.run(
            [OCR_PDF_COMMAND, '--skip-text', '--rotate-pages', '--deskew', '--output-type', 'pdf', str(pdf_path), str(ocr_pdf_path)],
            text=True,
            capture_output=True,
        )
        ocr_pdf_status = 'created' if result.returncode == 0 else f'failed: {result.stderr[-500:]}'
    elif requires_ocr_pdf:
        ocr_pdf_path, ocr_pdf_status = None, 'skipped: ocrmypdf_not_installed'
    else:
        ocr_pdf_path, ocr_pdf_status = None, 'not_needed'

    metadata = document.metadata or {}
    document_rows.append({
        'document_id': source_id,
        'category': category,
        'source_relative_path': str(relative_path),
        'source_sha256': source_hash,
        'source_bytes': pdf_path.stat().st_size,
        'page_count': len(document),
        'pdf_title': metadata.get('title', ''),
        'pdf_author': metadata.get('author', ''),
        'pdf_creator': metadata.get('creator', ''),
        'pdf_creation_date': metadata.get('creationDate', ''),
        'ocr_pdf_status': ocr_pdf_status,
        'ocr_pdf_file': str(ocr_pdf_path.relative_to(OUTPUT_ROOT)) if ocr_pdf_path and ocr_pdf_path.exists() else '',
    })
    document.close()

documents = pd.DataFrame(document_rows).sort_values(['category', 'source_relative_path'])
pages = pd.DataFrame(page_rows).sort_values(['source_relative_path', 'page_number'])
documents['duplicate_source_file'] = documents.groupby('source_sha256')['source_sha256'].transform('size').gt(1)
review_queue = pages[pages['review_required']].copy()
documents.to_csv(OUTPUT_ROOT / 'corpus_inventory.csv', index=False)
pages.to_csv(OUTPUT_ROOT / 'page_inventory.csv', index=False)
review_queue.to_csv(OUTPUT_ROOT / 'review_queue.csv', index=False)
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'source_root': str(SOURCE_ROOT),
    'document_count': len(documents),
    'page_count': len(pages),
    'ocr_page_count': int(pages['extraction_method'].eq('ocr').sum()),
    'review_page_count': len(review_queue),
    'searchable_ocr_pdfs_enabled': bool(OCR_PDF_COMMAND),
    'configuration': {'minimum_native_text_characters': MIN_NATIVE_TEXT_CHARS, 'ocr_dpi': OCR_DPI, 'ocr_language': OCR_LANGUAGE, 'low_confidence_threshold': LOW_CONFIDENCE},
}
(OUTPUT_ROOT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
display(documents[['category', 'source_relative_path', 'page_count', 'duplicate_source_file', 'ocr_pdf_status']])

Processing PDFs: 100%|██████████| 58/58 [11:58<00:00, 12.39s/it]


{
  "created_utc": "2026-07-17T13:09:32.793982+00:00",
  "source_root": "C:\\Users\\pchri\\Documents\\ChrisStuff\\WhitehouseDotGov",
  "document_count": 58,
  "page_count": 269,
  "ocr_page_count": 227,
  "review_page_count": 43,
  "searchable_ocr_pdfs_enabled": false,
  "configuration": {
    "minimum_native_text_characters": 75,
    "ocr_dpi": 300,
    "ocr_language": "eng",
    "low_confidence_threshold": 75.0
  }
}


,category,source_relative_path,page_count,duplicate_source_file,ocr_pdf_status
8,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,6,False,skipped: ocrmypdf_not_installed
9,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,24,False,skipped: ocrmypdf_not_installed
10,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,2,True,skipped: ocrmypdf_not_installed
11,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,1,True,skipped: ocrmypdf_not_installed
12,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,8,False,skipped: ocrmypdf_not_installed
13,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,6,False,skipped: ocrmypdf_not_installed
14,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,8,False,skipped: ocrmypdf_not_installed
15,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,24,False,skipped: ocrmypdf_not_installed
16,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,6,False,skipped: ocrmypdf_not_installed
17,china_voter_data,2. ChinaAcquisitionExploitationAmericanVoterDa...,8,True,not_needed


In [4]:
# Review low-yield and low-confidence pages against the original image before using their text.
if review_queue.empty:
    print('No pages were automatically queued for review. Sample-check OCR pages before substantive use.')
else:
    display(review_queue[[
        'source_relative_path',
        'page_number',
        'extraction_method',
        'ocr_average_confidence',
        'review_reasons',
        'text_file',
    ]])

# The next analysis notebook should cite every claim with:
# source_sha256 + source_relative_path + page_number + verbatim quote.
# Never treat the source's assertion as independently validated without corroboration.

,source_relative_path,page_number,extraction_method,ocr_average_confidence,review_reasons,text_file
0,1. Vulnerabilities in Voting and Counting Syst...,1,ocr,71.819444,low_ocr_confidence,page_text\cia_note_venezuela_machines_intel_me...
5,1. Vulnerabilities in Voting and Counting Syst...,6,ocr,21.000000,low_ocr_confidence;low_text_yield,page_text\cia_note_venezuela_machines_intel_me...
37,2. ChinaAcquisitionExploitationAmericanVoterDa...,1,ocr,71.968750,low_ocr_confidence,page_text\18_states_memo_clean_declass_mark_re...
38,2. ChinaAcquisitionExploitationAmericanVoterDa...,2,ocr,53.598985,low_ocr_confidence,page_text\18_states_memo_clean_declass_mark_re...
40,2. ChinaAcquisitionExploitationAmericanVoterDa...,4,ocr,66.440000,low_ocr_confidence,page_text\18_states_memo_clean_declass_mark_re...
41,2. ChinaAcquisitionExploitationAmericanVoterDa...,5,ocr,0.000000,low_ocr_confidence;low_text_yield,page_text\18_states_memo_clean_declass_mark_re...
44,2. ChinaAcquisitionExploitationAmericanVoterDa...,2,ocr,60.867647,low_ocr_confidence,page_text\200m_voter_records_compromised_decla...
45,2. ChinaAcquisitionExploitationAmericanVoterDa...,3,ocr,47.442060,low_ocr_confidence,page_text\200m_voter_records_compromised_decla...
65,2. ChinaAcquisitionExploitationAmericanVoterDa...,23,ocr,66.000000,low_ocr_confidence,page_text\200m_voter_records_compromised_decla...
66,2. ChinaAcquisitionExploitationAmericanVoterDa...,24,ocr,0.000000,low_ocr_confidence;low_text_yield,page_text\200m_voter_records_compromised_decla...
